# Assignment 4: A Cognitive FAQ System Using Pandas

## Question 1

In [ ]:
import pandas as pd
import re

roll_number_str = "1024170140"
categories = ["billing", "account", "general"]
fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.", "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings and choose Reset Password.", "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.", "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay by UPI, card, or net banking.", "keywords": "pay payment upi fee", "category": "billing"}
]
dynamic_entries = []
for digit in roll_number_str[-2:]:
    category = categories[int(digit) % 3]
    if category == "billing":
        dynamic_entries.append({"question": "how can i check my bill", "answer": "Open the billing section to view your bill.", "keywords": "bill invoice billing", "category": category})
    elif category == "account":
        dynamic_entries.append({"question": "how do i update my mobile number", "answer": "Open account settings and update your mobile number.", "keywords": "mobile number account update", "category": category})
    else:
        dynamic_entries.append({"question": "where can i get help", "answer": "Contact the help desk for assistance.", "keywords": "help support assistance", "category": category})
df_faq = pd.DataFrame(fixed_entries + dynamic_entries)
display(df_faq)

## Question 2

In [ ]:
def score_faq_entries(query, df):
    query_words = set(re.findall(r"[a-z0-9]+", query.lower()))
    results = []
    for index, row in df.iterrows():
        text = " ".join(str(row[col]) for col in ["question", "answer", "keywords"]).lower()
        words = set(re.findall(r"[a-z0-9]+", text))
        score = len(query_words & words)
        results.append({"question": row["question"], "answer": row["answer"], "category": row["category"], "confidence_score": score})
    return pd.DataFrame(results).sort_values("confidence_score", ascending=False).reset_index(drop=True)

query = input("Enter a question: ")
display(score_faq_entries(query, df_faq))

## Question 3

In [ ]:
def same_category(category_name, df):
    return df[df["category"].str.lower() == category_name.lower()]

chosen_category = dynamic_entries[0]["category"]
display(same_category(chosen_category, df_faq))

## Question 4

In [ ]:
entry_index = 0
new_keyword = input("Enter a keyword: ").strip()
if new_keyword:
    keywords = df_faq.at[entry_index, "keywords"].split()
    if new_keyword.lower() not in [word.lower() for word in keywords]:
        keywords.append(new_keyword)
    df_faq.at[entry_index, "keywords"] = " ".join(keywords)
df_faq.to_csv(roll_number_str + "_faq_data.csv", index=False)
display(df_faq)

## Question 5

In [ ]:
print(df_faq.groupby("category").size())

## Question 6

In [ ]:
def show_best_matches(query, df):
    ranked = score_faq_entries(query, df)
    if ranked.empty:
        print("No entries found")
        return
    best_score = ranked["confidence_score"].max()
    best_matches = ranked[ranked["confidence_score"] == best_score]
    if best_score == 0:
        print("No matching entries found")
    else:
        display(best_matches)

show_best_matches("fee", df_faq)
show_best_matches("reset password", df_faq)